# Week 3 Homework — Wikipedia Agent with ChatGPT

Build an agent that can search Wikipedia and fetch page content to answer questions.

In [1]:
%pip install openai requests --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import requests
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"

HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

## Question 1 & 2 — Search Tool

In [4]:
def search_wikipedia(query: str) -> dict:
    """Search Wikipedia and return results for a query."""
    url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return response.json()


results = search_wikipedia("capybara")
search_items = results["query"]["search"]

print(f"Total results: {results['query']['searchinfo']['totalhits']}")
print(f"Results returned: {len(search_items)}")
print("\nTitles:")
for item in search_items:
    print(" -", item["title"])

Total results: 910
Results returned: 10

Titles:
 - Capybara
 - Lesser capybara
 - Capybara (disambiguation)
 - Capybara Games
 - Hydrochoerus
 - Capybara (software)
 - Caviidae
 - Flow (2024 film)
 - Yuzu bath
 - Kerodon


In [5]:
# Q2: Count titles containing 'capybara' (case-insensitive)
capybara_titles = [item for item in search_items if "capybara" in item["title"].lower()]
print(f"Titles containing 'capybara': {len(capybara_titles)}")
for item in capybara_titles:
    print(" -", item["title"])

Titles containing 'capybara': 5
 - Capybara
 - Lesser capybara
 - Capybara (disambiguation)
 - Capybara Games
 - Capybara (software)


## Question 3 — Get Page Tool

In [6]:
def get_page(title: str) -> str:
    """Fetch raw content of a Wikipedia page by title."""
    url = f"https://en.wikipedia.org/w/index.php?title={title}&action=raw"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return response.text


capybara_page = get_page("Capybara")
print(f"Characters in Capybara page: {len(capybara_page):,}")
print("\nFirst 500 chars:")
print(capybara_page[:500])

Characters in Capybara page: 36,946

First 500 chars:
{{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=''Hydrochoerus hydrochaeris'' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>
| image = Capybaracropped.jpg
| image_caption = In [[Encontro das Águas State Park]], Brazil
| genus = Hydrochoer


## Question 4 — Agent Setup

- **Framework**: Custom agentic loop (from scratch)
- **LLM provider**: OpenAI (`gpt-4o-mini`)

In [7]:
# Tool schemas for OpenAI function calling
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Search Wikipedia for pages related to a topic. Returns a list of matching page titles and snippets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query (use + for spaces, e.g. 'lesser+capybara')",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_page",
            "description": "Fetch the full raw content of a Wikipedia page by its exact title.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The exact Wikipedia page title (e.g. 'Capybara')",
                    }
                },
                "required": ["title"],
            },
        },
    },
]


def call_tool(name: str, arguments: dict) -> str:
    """Dispatch a tool call and return the result as a string."""
    if name == "search_wikipedia":
        data = search_wikipedia(arguments["query"])
        items = data["query"]["search"]
        return json.dumps([{"title": i["title"], "snippet": i["snippet"]} for i in items])
    elif name == "get_page":
        content = get_page(arguments["title"])
        # Truncate to ~8000 chars to stay within context limits
        return content[:8000]
    else:
        return f"Unknown tool: {name}"


def run_agent(user_question: str, verbose: bool = True) -> str:
    """Run the agentic loop until the model produces a final answer."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful research assistant with access to Wikipedia. "
                "Use the provided tools to look up accurate information before answering."
            ),
        },
        {"role": "user", "content": user_question},
    ]

    tool_call_count = 0

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            if verbose:
                print(f"\nTotal tool calls made: {tool_call_count}")
            return message.content

        for tc in message.tool_calls:
            tool_call_count += 1
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"[Tool call #{tool_call_count}] {tc.function.name}({args})")

            result = call_tool(tc.function.name, args)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                }
            )

print("Agent ready.")

Agent ready.


## Question 5 — Single Page Fetch

In [8]:
q5 = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
answer5 = run_agent(q5)
print("\nAgent answer:")
print(answer5)

[Tool call #1] get_page({'title': 'Capybara'})

Total tool calls made: 1

Agent answer:
The Wikipedia page on the capybara provides detailed information about this remarkable rodent, the largest living rodent species. The scientific name for the capybara is *Hydrochoerus hydrochaeris*, and it is native to all South American countries except Chile. Here are some key points covered in the article:

1. **Habitat and Diet**: Capybaras are semiaquatic herbivores typically found in savannas and dense forests near water bodies. Their diet primarily consists of grasses and aquatic plants.

2. **Social Behavior**: They are highly social animals, often living in groups ranging from 10 to 20, but can form larger groups of up to 100 individuals.

3. **Etymology**: The name "capybara" is derived from the Tupi language, reflecting its dietary habits. The genus name *Hydrochoerus* combines Greek words meaning "water" and "pig."

4. **Classification**: Capybaras belong to the genus *Hydrochoerus*, and

## Question 6 — Search Then Fetch

In [9]:
q6 = "What are the main threats to capybara populations?"
answer6 = run_agent(q6)
print("\nAgent answer:")
print(answer6)

[Tool call #1] search_wikipedia({'query': 'capybara threats'})
[Tool call #2] get_page({'title': 'Capybara'})

Total tool calls made: 2

Agent answer:
Capybara populations face several main threats:

1. **Hunting**: Capybaras are hunted for their meat, hide, and fat, which are valued in various cultures. This hunting pressure can lead to population declines, particularly in areas where hunting is not regulated.

2. **Habitat Loss**: As human populations expand, many capybara habitats are being converted for agriculture, urban development, and infrastructure projects. This results in a reduction of their natural habitats, particularly near bodies of water where they thrive.

3. **Climate Change**: Changes in climate can affect the availability of water sources and the quality of their habitats. Droughts or flooding can have significant impacts on their survival and reproduction.

4. **Predation**: While capybaras have natural predators such as jaguars and caimans, increased predation pr